In [ ]:

# ============================================
# TOMATO PLANT LEAF DISEASE DETECTION - COLAB
# Run everything in ONE cell
# ============================================

!pip -q install flask pyngrok torch torchvision pillow

import os
import subprocess
import threading
import time
from google.colab import files
from pyngrok import ngrok

# 1. Upload app.py, cnn_model.pth, and a ZIP containing templates/
#    (including index.html) and any other required files.
print("Upload app.py, cnn_model.pth, and your templates/static ZIP if available.")
uploaded = files.upload()

# 2. Extract uploaded ZIP files
import zipfile
for filename in uploaded:
    if filename.lower().endswith(".zip"):
        with zipfile.ZipFile(filename, "r") as archive:
            archive.extractall("/content")
        print("Extracted:", filename)

# 3. Check required files
app_path = "/content/app.py"
model_path = "/content/cnn_model.pth"

if not os.path.exists(app_path):
    raise FileNotFoundError("Upload your Flask Python file and name it app.py.")

if not os.path.exists(model_path):
    raise FileNotFoundError("Upload your trained model named cnn_model.pth.")

if not os.path.exists("/content/templates/index.html"):
    raise FileNotFoundError(
        "Missing templates/index.html. Upload a ZIP containing "
        "templates/index.html and your static files."
    )

# 4. IMPORTANT: Update the dataset_dir inside app.py to a valid
#    Colab/Google Drive path. The original G:\\ path won't work in Colab.
#    Your training class folders must match the model's class order.

# 5. Configure ngrok
# Get your token at https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = input("Paste your ngrok auth token: ").strip()

if not NGROK_AUTH_TOKEN:
    raise ValueError("An ngrok auth token is required.")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()

# 6. Start Flask
process = subprocess.Popen(
    ["python", app_path],
    cwd="/content",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# 7. Create public URL
try:
    public_url = ngrok.connect(5000).public_url
    print("\\nYour Tomato Disease Detection website:")
    print(public_url)
    print("\\nKeep this Colab cell running while you use the website.")
except Exception:
    process.terminate()
    raise